# MedMNIST-C API debug

This notebook verifies the cloned API and its data flow. The smoke dataset is resized from two official 28-pixel PathMNIST test images only to exercise the API; its outputs must never be reported as robustness results. Full generation requires official `pathmnist_224.npz` or the corresponding official 224 files.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil
import numpy as np
from PIL import Image

ROOT = Path('/project/prj-sis01/xuxiaoyu/reliability_medmnistc_ab')
REPO = ROOT / 'sources' / 'medmnistc-api'
SMOKE_CLEAN = ROOT / 'data' / 'medmnist_smoke'
SMOKE_CORRUPTED = ROOT / 'data' / 'medmnistc_smoke'
SMOKE_RESULTS = ROOT / 'results' / 'raw_metrics' / 'api_smoke'
SMOKE_CLEAN.mkdir(parents=True, exist_ok=True); SMOKE_CORRUPTED.mkdir(parents=True, exist_ok=True); SMOKE_RESULTS.mkdir(parents=True, exist_ok=True)
print(REPO)

In [ ]:
import os, subprocess, sys
sys.dont_write_bytecode = True
assert (REPO / 'medmnistc' / 'dataset_manager.py').exists()
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--short'], text=True)
code_dirty = [line for line in status.splitlines() if '__pycache__' not in line]
assert not code_dirty, code_dirty
print({'commit': commit, 'python': sys.version, 'source_code_clean': True, 'cache_status_before_import': status.splitlines()})

In [ ]:
from medmnist import PathMNIST
source_root = Path('/project/prj-sis01/xuxiaoyu/HOP/datasets/PathMNIST')
source = np.load(source_root / 'pathmnist.npz')
imgs = source['test_images'][:2]
labels = source['test_labels'][:2]
resized = np.stack([np.asarray(Image.fromarray(x).resize((224, 224), Image.Resampling.BILINEAR)) for x in imgs])
np.savez_compressed(SMOKE_CLEAN / 'pathmnist_224.npz', train_images=resized, val_images=resized, test_images=resized, train_labels=labels, val_labels=labels, test_labels=labels)
print({'synthetic_smoke_shape': resized.shape, 'source_shape': imgs.shape})

In [ ]:
from medmnistc.dataset_manager import DatasetManager
from medmnistc.dataset import CorruptedMedMNIST
from medmnistc.corruptions.registry import CORRUPTIONS_DS
manager = DatasetManager(medmnist_path=str(SMOKE_CLEAN), output_path=str(SMOKE_CORRUPTED), random_seed=0)
manager.create_dataset('pathmnist')
corrs = list(CORRUPTIONS_DS['pathmnist'])
checks = {}
for corruption in corrs:
    ds = CorruptedMedMNIST('pathmnist', corruption, root=str(SMOKE_CORRUPTED), as_rgb=True)
    checks[corruption] = {'length': len(ds), 'sample_shape': list(ds[0][0].shape), 'labels': int(len(ds.labels))}
summary = {'kind': 'synthetic_api_smoke_only', 'corruptions': checks, 'source_commit': commit}
(SMOKE_RESULTS / 'medmnistc_api_smoke_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)